In [1]:
import os

RAW_DIR = "/mnt/d/matics/logs"
images_dirs = [
    os.path.join(RAW_DIR, "coco", "images"),
    os.path.join(RAW_DIR, "processed", "batch_01", "images"),
]
json_path = os.path.join(RAW_DIR, "processed", "batch_01", "annotations.json")

In [2]:
print(len(os.listdir(images_dirs[1])))

400


In [ ]:
from pycocotools.coco import COCO
import json


def join_coco_batches(images_dirs: list[str], json_path: str, output_dir: str):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    coco = COCO(json_path)

    # Remove images that have no annotations from dataset, store new json
    coco_images = coco.loadImgs(coco.getImgIds())
    coco_images = [
        img for img in coco_images if len(coco.getAnnIds(imgIds=img["id"])) > 0
    ]
    # Print number of images before and after
    print(f"Number of images before: {len(coco.dataset['images'])}")
    print(f"Number of images after: {len(coco_images)}")
    coco.dataset["images"] = coco_images
    # check there are no annotations without images, if there are raise an error
    coco_anns = coco.loadAnns(coco.getAnnIds())
    coco_anns_checked = [ann for ann in coco_anns if coco.loadImgs(ann["image_id"])]
    if len(coco_anns) != len(coco_anns_checked):
        raise ValueError("Annotations without corresponding images found")
    # save new json
    with open(os.path.join(output_dir, "annotations.json"), "w") as f:
        json.dump(coco.dataset, f)

    images_filenames = [img["file_name"] for img in coco_images]
    for images_dir in images_dirs:
        for filename in os.listdir(images_dir):
            if filename in images_filenames:
                src = os.path.join(images_dir, filename)
                dst = os.path.join(output_dir, "images", filename)
                # copy
                os.system(f"cp {src} {dst}")

In [9]:
join_coco_batches(images_dirs, json_path, ".")

loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
Number of images before: 800
Number of images after: 600
